# Programmatic Magnetic Resonance Fingerprinting Optimization

### Orthogonality 📐

The accuracy and reliability of parameter estimation in MRF fundamentally depend on how distinguishable different signal fingerprints are from one another, leading to the critical aspect of signal orthogonality. When signal evolutions corresponding to different tissue parameters are highly orthogonal (i.e., minimally correlated), the matching process can more reliably distinguish between tissues with similar properties. Lower signal orthogonality for signal fingerprints with different relaxometric origins can lead to increased parameter estimation errors and reduced robustness to noise.


### Cramér-Rao Lower Bound (CRLB) ⛓

The Cramér-Rao Lower Bound (CRLB) provides a theoretical framework for understanding the best possible precision achievable in parameter estimation. In the context of MRF, the CRLB quantifies the minimum variance (uncertainty) in estimating parameters from the signal vectors. The inverse of the Fisher Information Matrix yields the CRLB, with diagonal elements representing the variance lower bounds for each parameter. Importantly, off-diagonal elements reveal correlations between parameter estimates. When parameters are highly correlated, they cannot be estimated independently with high precision.

### Forward Model Extended Phase Graph (EPG) 🌀

To optimize MRF acquisition parameters, we require a forward model that can accurately simulate the complex signal evolution during the sequence. The Extended Phase Graph (EPG, see e.g. [Weigel 2015](https://doi.org/10.1002/jmri.24619)) formalism provides an elegant and computationally efficient solution for this purpose.

# Programmatic Magnetic Resonance Fingerprinting Optimization

### Orthogonality 📐

The accuracy and reliability of parameter estimation in MRF fundamentally depend on how distinguishable different signal fingerprints are from one another, leading to the critical aspect of signal orthogonality. When signal evolutions corresponding to different tissue parameters are highly orthogonal (i.e., minimally correlated), the matching process can more reliably distinguish between tissues with similar properties. Lower signal orthogonality for signal fingerprints with different relaxometric origins can lead to increased parameter estimation errors and reduced robustness to noise.


### Cramér-Rao Lower Bound (CRLB) ⛓

The Cramér-Rao Lower Bound (CRLB) provides a theoretical framework for understanding the best possible precision achievable in parameter estimation. In the context of MRF, the CRLB quantifies the minimum variance (uncertainty) in estimating parameters from the signal vectors. The inverse of the Fisher Information Matrix yields the CRLB, with diagonal elements representing the variance lower bounds for each parameter. Importantly, off-diagonal elements reveal correlations between parameter estimates. When parameters are highly correlated, they cannot be estimated independently with high precision.

### Forward Model Extended Phase Graph (EPG) 🌀

To optimize MRF acquisition parameters, we require a forward model that can accurately simulate the complex signal evolution during the sequence. The Extended Phase Graph (EPG, see e.g. [Weigel 2015](https://doi.org/10.1002/jmri.24619)) formalism provides an elegant and computationally efficient solution for this purpose.

# Programmatic Magnetic Resonance Fingerprinting Optimization

### Orthogonality 📐

The accuracy and reliability of parameter estimation in MRF fundamentally depend on how distinguishable different signal fingerprints are from one another, leading to the critical aspect of signal orthogonality. When signal evolutions corresponding to different tissue parameters are highly orthogonal (i.e., minimally correlated), the matching process can more reliably distinguish between tissues with similar properties. Lower signal orthogonality for signal fingerprints with different relaxometric origins can lead to increased parameter estimation errors and reduced robustness to noise.


### Cramér-Rao Lower Bound (CRLB) ⛓

The Cramér-Rao Lower Bound (CRLB) provides a theoretical framework for understanding the best possible precision achievable in parameter estimation. In the context of MRF, the CRLB quantifies the minimum variance (uncertainty) in estimating parameters from the signal vectors. The inverse of the Fisher Information Matrix yields the CRLB, with diagonal elements representing the variance lower bounds for each parameter. Importantly, off-diagonal elements reveal correlations between parameter estimates. When parameters are highly correlated, they cannot be estimated independently with high precision.

### Forward Model Extended Phase Graph (EPG) 🌀

To optimize MRF acquisition parameters, we require a forward model that can accurately simulate the complex signal evolution during the sequence. The Extended Phase Graph (EPG, see e.g. [Weigel 2015](https://doi.org/10.1002/jmri.24619)) formalism provides an elegant and computationally efficient solution for this purpose.

First we set up the necessary code and jupyter environment.

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as wgt
import tqdm.auto as tqdm
import rich
import torch

from IPython.display import display
from pathlib import Path
from numpy.typing import NDArray
from typing import Literal, List, Union
from collections.abc import Sequence

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

This should be readily importable when the repository is cloned from GitHub and the notebook is run in its repository defined folder 🚀

In [3]:
src_directory = Path('../src')
init_directory = src_directory / 'initialization'
sys.path.append(str(src_directory))
print = rich.print # nicer outputs

In [4]:
import seqmetrics # noqa: F401
from plotting.splinetools import SplineSettingsDashboard
from plotting.curveditor import MonotonicCurveEditor
from plotting.simulator import SimulationController, SimulationControllerDashboard
from plotting.scattercanvas import InteractiveRelaxometricParameterCanvas
from plotting.signaldisplay import SignalDisplayDashboard, SignalDisplay, wire_callbacks
from plotting.historyplot import HistoryPlot, PrecomputedOptimizationPlot
from plotting.sequence import SequenceParameters
from slsqp import optimize_sequence
from tools import OptimizationPackage

Here we load the ncessary flip angle train data and the repetition time pattern from preset numpy arrays.
Both patterns were deduced from the brain-specific work by [Cao et al. 2022](https://doi.org/10.1002/mrm.29194).
They act as an starting point for the interactive modification tooling and subsequently the optimization.
In principle, any other patterns can be loaded and tested here.
Other examples include the:
- original sinusoidal pattern by [Yun et al. 2015](https://doi.org/10.1002/mrm.25559)
- optimized patterns by [Zhao et al. 2019](10.1109/TMI.2018.2873704)

In [5]:
FA_DATA_PATH = init_directory / 'fa_cao.npy'
TR_DATA_PATH = init_directory / 'tr_cao.npy'
fa = np.load(FA_DATA_PATH)
tr = np.load(TR_DATA_PATH)
fa_initial_y = fa
fa_initial_x = np.arange(len(fa))
tr_initial_y = tr
tr_initial_x = np.arange(len(tr))

The interactive spline editors provide an intuitive way to manipulate MR fingerprinting sequences through direct manual adjustment of acquisition parameters.

Flip Angle (FA) Editor:
- Purpose: Design flip angle trajectories across the sequence timepoints
- Controls: Click and drag control points to shape the FA curve
- Range: Typically 0-90 degrees for optimal signal variation
- Tips:
  - Periodic patterns can enhance tissue discrimination
  - Sharp transitions create distinct signal signatures
  - Use reset inside tabs to revert to initial state

Repetition Time (TR) Editor:
- Purpose: Define variable TR patterns for enhanced parameter encoding
- Controls: Click and drag control points to shape the TR curve
- Range: Usually 10-50 ms for rapid fingerprinting sequences
- Benefits:
  - Variable TR improves T1/T2 sensitivity separation
  - Shorter TRs enable faster acquisition

Scatter Canvas Editor
- Purpose: Define relaxometric species
- Controls
  - Leftclick into (T1, T2) grid to add new relaxometric species
  - Rightclick deletes closest species in (T1,T2) grid
  - Species with defined (T1,T2) combinations are automatically utilized in subsequent simulations

In [6]:
dashboard_fa = SplineSettingsDashboard.from_FA_defaults()
dashboard_tr = SplineSettingsDashboard.from_TR_defaults()

seqparams = SequenceParameters(
    ph=np.full_like(fa, fill_value=0.0),
    shots=len(fa),
    prep=[1],
    t2te=[0.0],
    ti=[10.0],
    te=1.0
)

with plt.ioff():
    fig, axes = plt.subplots(ncols=3, figsize=(13.3, 3.9))
    ce_fa = MonotonicCurveEditor(fa_initial_x, fa_initial_y, dashboard_fa, fig=fig, ax=axes[0])
    ce_fa.ax.set_ylabel('Flip Angle (degrees)')
    ce_fa.ax.set_title('Interactive Flip Angle Editor')

    ce_tr = MonotonicCurveEditor(tr_initial_x, tr_initial_y, dashboard_tr, fig=fig, ax=axes[1], initial_yaxis_range=(0, 100))
    ce_tr.ax.set_ylabel('Repetition Time (ms)')
    ce_tr.ax.set_title('Interactive Repetition Time Editor')

    cv = InteractiveRelaxometricParameterCanvas.prepopulated(species={'csf', 'wm', 'gm', 'muscle'}, ax=axes[2], fig=fig)

    ctr_dashboard = SimulationControllerDashboard()

    controller = SimulationController(
        dashboard=ctr_dashboard,
        fa_provider=ce_fa,
        tr_provider=ce_tr,
        relax_provider=cv,
        parameters=seqparams
    )

    tabs = wgt.Tab(
        children=[ctr_dashboard.ui, dashboard_fa.ui, dashboard_tr.ui],
        titles=['Simulation Dashboard', 'FA Settings', 'TR Settings'],
        style={'description_width': 'initial'}
    )

    _ = fig.tight_layout()
    
controller.run()

In [7]:
dashboard = SignalDisplayDashboard.create()
sigdisp = SignalDisplay()
sigdisp.fig.update_layout(height=400, width=1400, showlegend=False)
sigdisp.fig.update_layout(xaxis=dict(title='Time (ms)'), yaxis=dict(title='Signal Amplitude (a.u.)'))

sigdisp.add_traces(controller.fetch_simulation_package(), dashboard.query_state())
wire_callbacks(sigdisp, dashboard, controller)

In [8]:
display(
    wgt.VBox([
        wgt.VBox([tabs, fig.canvas]),
        dashboard.ui,
        sigdisp.fig
    ])
)

With the setup about the relaxometric species from the scatter canvas and the sequence specific parameters {flip angle train, repetition times} from the interactive spline curve obtained above, we can optimize the sequence with respect to different cost functions.

The parameter data defined by the points inside the relaxometric species canvas is reused for the optimization.
Note that the optimization does depend on the currently set species, since the output signal fitness is quantified via the orthogonality cost function.
In turn, this means that we can optimize for specific anatomic regions with prior knowledge about expected tissue
environments and subsequent relaxometric parameters.

In [9]:
T1: NDArray = np.asarray([p.x for p in cv.get_points()], dtype=np.float32)
T2: NDArray = np.asarray([p.y for p in cv.get_points()], dtype=np.float32)
M0: float = 1.0

In [10]:
# these are some parameters needed for the optimization
# when we utilize the more information-theoretical approach
# with the Cramer-Rao bound.
# For the default demo with orthogonality, these are not used.
ratio: float | None = None
weighting: Sequence[float] | None = [1/T1, 1/T2, 1/M0]

We also utilize the sequence data from the interactive editors above as a starting point the optimization.

In [11]:
fa: NDArray = ce_fa.get_current_curve().y
tr: NDArray = ce_tr.get_current_curve().y
ph: float = seqparams.ph

SLSQP support simple box constraints on variables. We utilize this for the flip angle like so: $5^{\circ} \leq \alpha \leq 90^{\circ}$ and $\Delta_{\mathrm{alpha}}=5^{\circ}$. Also the maximum number of iterations determines the overall optimization wall clock duration.

In [12]:
n_max_iter: int = 5
fa_min: float = 5.0
fa_max: float = 90.0
fa_maxdiff: float = 5.0
cost_function: Literal['crlb_sc_iso', 'crlb_sc_epg', 'crlb_mc_iso', 'crlb_mc_epg', 'orth_iso', 'orth_epg'] = 'orth_epg'

In [13]:
# Other sequence parameters (already defined above, but extracted here for clarity)
beats: int = seqparams.beats
shots: int = seqparams.shots
prep: list[int] = seqparams.prep
ti: list[float] = seqparams.ti
t2te: list[float] = seqparams.t2te
te: float = seqparams.te
ph: NDArray = seqparams.ph

With all the parameters in place, we can now run the optimization with respect to the orthogonality criterion.
This criterion attempts to minimize the inner product of the signal vectors of the relaxometric species via SLSQP and automatic differentiation-based computation of the gradient.
Decreased correlation between fingerprints is expected to improve matching accuracy and increase noise robustness.

Below we see the raw canvas of the live updated optimization history plot that shows the last `maximum_time_to_live` flip angle trains of the optimization process.
The optimization process is run automatically for 10 iterations (see two cells above if you want more iterations) when we run the cell below the plot.
A progress bar under this cell also shows the progress of the iterative optimization.

In [17]:
maximum_time_to_live: int = 10

hp = HistoryPlot(max_TTL=maximum_time_to_live)
init_fa = ce_fa.get_current_curve().y
hp.add_immortal_trace(init_fa, width=1, color='green', dash='dot', opacity=0.7)
hp.fig.update_layout(xaxis=dict(title='TR index'), yaxis=dict(title='Flip Angle (degrees)'))

FigureWidget({
    'data': [{'line': {'color': 'green', 'dash': 'dot', 'width': 1},
              'meta': {'TTL': -999, 'is_immortal': True},
              'mode': 'lines',
              'name': 'immortal',
              'opacity': 0.7,
              'type': 'scatter',
              'uid': '9d89cab5-3a31-4f19-b631-e4aa5b5663a7',
              'y': {'bdata': ('MDHd+rOiJEB76BX5LZcrQNcf10s+QD' ... '/0aUhAa843Au3ESUBXxGCf7yBLQA=='),
                    'dtype': 'f8'}}],
    'layout': {'height': 400,
               'showlegend': True,
               'template': '...',
               'width': 1200,
               'xaxis': {'title': {'text': 'TR index'}},
               'yaxis': {'title': {'text': 'Flip Angle (degrees)'}}}
})

In [18]:
pbar = tqdm.tqdm(total=n_max_iter)
opthistory = []

def callback(x: np.ndarray):
    # tgriesler optimizatiopn: single array for fa and tr
    # only propagate flip angles into visualization
    data = x[:x.size//2]
    hp.add_trace(data)
    opthistory.append(data)
    pbar.update(1)

result = optimize_sequence(
    costfunction=cost_function,
    t1=T1,
    t2=T2,
    m0=M0,
    beats=beats,
    shots=shots,
    fa=init_fa,
    tr=tr,
    ph=ph,
    prep=prep,
    ratio=ratio,
    weighting=weighting,
    ti=ti,
    t2te=t2te,
    te=te,
    fa_min=fa_min,
    fa_max=fa_max,
    fa_maxdiff=fa_maxdiff,
    n_iter_max=n_max_iter,
    callback=callback,
    iprint=0
)

  0%|          | 0/5 [00:00<?, ?it/s]

Since this optimization process can take quite a while, we prepared some longer running optimizations as packaged data for easy viewing.
We precomputed three optimization scenarios with the cost functions:
- CRLB MC EPG: Cramér-Rao Lower Bound for Multi-Compartment model using EPG signal model
- CRLB SC EPG: Cramér-Rao Lower Bound for Single-Compartment model using EPG signal model
- Orthogonality EPG: Orthogonality between signal vectors of relaxometric species for EPG signal model

We can select them via the buttons in the below interactive figure. Due to the large number of optimization steps, plotting and visualization can sometimes be slow, depending on your system resources, viewer software and the selected range of optimization steps concurrently displayed. If you see any visual glitches, play around with the toggle buttons or wait a few seconds until the visualization loop has caught up to the UI input 🤓

In [16]:
pcplot = PrecomputedOptimizationPlot(None)
ui = wgt.HBox([pcplot.runselector, pcplot.plot.rangeselector, pcplot.opacityselector])
display(wgt.VBox([ui, pcplot.plot.fig]))